# 🌍 World Happiness Report 2025 — Comprehensive Analysis Notebook

---

## Table of Contents

**Part I — Project Overview & Methodology**
1. [Introduction & Motivation](#1-introduction--motivation)
2. [Dataset Description & Source](#2-dataset-description--source)
3. [Libraries Used & Why](#3-libraries-used--why)
4. [Analysis Pipeline Overview](#4-analysis-pipeline-overview)
5. [Design Decisions & Rationale](#5-design-decisions--rationale)
6. [Chart-by-Chart Methodology](#6-chart-by-chart-methodology)

**Part II — Step-by-Step Code Walkthrough**
7. [Step 1: Imports & Environment Setup](#step-1-imports--environment-setup)
8. [Step 2: Global Matplotlib Theme](#step-2-global-matplotlib-theme)
9. [Step 3: Region Color Palette](#step-3-region-color-palette)
10. [Step 4: Country-to-Region Mapping](#step-4-country-to-region-mapping)
11. [Step 5: Data Loading & Preparation](#step-5-data-loading--preparation)
12. [Step 6: Türkiye Reference Point](#step-6-türkiye-reference-point)
13. [Step 7: Chart 1 — Top 20 & Bottom 20 Rankings](#step-7-chart-1--top-20--bottom-20-rankings)
14. [Step 8: Chart 2 — Factor Breakdown (Top 30)](#step-8-chart-2--factor-breakdown-top-30)
15. [Step 9: Chart 3 — Regional Distribution](#step-9-chart-3--regional-distribution)
16. [Step 10: Chart 4 — GDP vs Happiness Scatter](#step-10-chart-4--gdp-vs-happiness-scatter)
17. [Step 11: Chart 5 — Factor Correlations](#step-11-chart-5--factor-correlations)
18. [Step 12: Chart 6 — Happiness Paradox](#step-12-chart-6--happiness-paradox)

---

# PART I — PROJECT OVERVIEW & METHODOLOGY

---

## 1. Introduction & Motivation

The **World Happiness Report (WHR)** is an annual publication by the United Nations Sustainable Development Solutions Network. It ranks countries by how happy their citizens perceive themselves to be, using data from the **Gallup World Poll**. Respondents rate their own lives on a scale from 0 (worst possible life) to 10 (best possible life) — this is called the **Cantril Self-Anchoring Scale**, or simply the "Ladder Score."

### Why This Analysis?

The 2025 edition covers **147 countries** and decomposes each nation's happiness score into six measurable factors:

| Factor | What It Measures |
|--------|------------------|
| **Log GDP per Capita** | Economic output per person (logarithmic scale) |
| **Social Support** | Having someone to count on in times of trouble |
| **Healthy Life Expectancy** | Average years of healthy life expected at birth |
| **Freedom to Make Life Choices** | Satisfaction with personal freedom |
| **Generosity** | Recent charitable donations (adjusted for GDP) |
| **Perceptions of Corruption** | Belief that corruption is widespread in government/business |

Our goal is to go beyond simple rankings and answer deeper questions:

- What **drives** happiness in the top-ranked countries?
- How do **regions** of the world compare?
- Is the relationship between **wealth and happiness** truly linear?
- Which factor has the **strongest statistical correlation** with happiness?
- Are there countries that are significantly **happier or unhappier than their GDP predicts**? (The "Happiness Paradox")

We produce **6 publication-quality visualizations** to answer each of these questions.

## 2. Dataset Description & Source

### Source

The dataset is sourced from **Kaggle**: [World Happiness Report 2025](https://www.kaggle.com/datasets/rmarbun/world-happiness-report-2025/data), published by user **rmarbun**. The underlying data originates from the official [World Happiness Report](https://worldhappiness.report/) and the **Gallup World Poll**.

### File Details

| Property | Value |
|----------|-------|
| **Filename** | `WHR_2025.xls` |
| **Format** | Legacy Excel `.xls` (BIFF format) |
| **Rows** | 147 (one per country) |
| **File Size** | ~194 KB |

### Column Descriptions

| Original Column Name | Our Renamed Column | Description |
|----------------------|-------------------|-------------|
| `Country name` | `country` | Official country name |
| `Ladder score` | `happiness` | Overall happiness score (0–10 scale) |
| `upperwhisker` | `upper` | Upper bound of 95% confidence interval |
| `lowerwhisker` | `lower` | Lower bound of 95% confidence interval |
| `Explained by: Log GDP per capita` | `gdp` | GDP contribution to the happiness score |
| `Explained by: Social support` | `social` | Social support contribution |
| `Explained by: Healthy life expectancy` | `life_exp` | Life expectancy contribution |
| `Explained by: Freedom to make life choices` | `freedom` | Freedom contribution |
| `Explained by: Generosity` | `generosity` | Generosity contribution |
| `Explained by: Perceptions of corruption` | `corruption` | Low-corruption contribution |
| `Dystopia + residual` | `dystopia` | Baseline score + unexplained variance |

### Why `.xls` and not `.csv`?

The Kaggle dataset is distributed in legacy Excel `.xls` format (not the modern `.xlsx`). This means we need the `xlrd` library specifically — `openpyxl` cannot read this older binary format. This is a common nuance in real-world data work.

### What is "Dystopia + Residual"?

The WHR methodology defines a hypothetical "Dystopia" — the world's unhappiest possible country. Each real country's score is the Dystopia baseline **plus** the contributions from each factor **plus** a residual term (unexplained variance). The `dystopia` column captures this baseline + residual combined.

## 3. Libraries Used & Why

Every library in this project was chosen for a specific reason. Here's the rationale:

### Core Libraries

| Library | Version | Purpose | Why This Library? |
|---------|---------|---------|-------------------|
| **`pandas`** | latest | Data loading, wrangling, transformation | The gold standard for tabular data in Python. Its DataFrame API makes column renaming, filtering, grouping, and sorting intuitive and fast. No serious data analysis in Python is done without it. |
| **`numpy`** | latest | Numerical operations, array math | Required for vectorized operations (e.g., `np.zeros` for stacked bar offsets, `np.polyfit` for linear regression, `np.linspace` for trend lines). Pandas itself is built on top of NumPy. |
| **`matplotlib`** | latest | Chart generation & rendering | The most mature, flexible plotting library in Python. Unlike Seaborn or Plotly, matplotlib gives us pixel-level control over every element — essential for publication-quality charts with custom badges, annotations, and multi-panel layouts. |
| **`matplotlib.patches`** | (submodule) | Custom legend patches | Used to create colored rectangle patches for our region legend, since we color bars programmatically rather than using matplotlib's auto-legend. |
| **`scipy.stats`** | latest | Statistical testing | Provides `pearsonr()` for computing Pearson correlation coefficients and their p-values, and `zscore()` for standardization in the paradox analysis. SciPy is the standard scientific computing library. |
| **`xlrd`** | latest | Reading `.xls` files | The only Python library that can read legacy BIFF-format `.xls` files. `openpyxl` only handles `.xlsx`. Since our Kaggle dataset is in `.xls` format, `xlrd` is required. |

### Utility Libraries

| Library | Purpose | Why? |
|---------|---------|------|
| **`os`** | File system operations | Used only for `os.makedirs('results', exist_ok=True)` — ensures the output directory exists before saving charts. |
| **`sys`** | System configuration | Used to force `stdout` encoding to UTF-8 on Windows, preventing `UnicodeEncodeError` when printing emoji characters (🇹🇷) in Turkish-locale terminals. |
| **`warnings`** | Warning suppression | Matplotlib generates non-critical warnings (font fallbacks, deprecation notices) that clutter the output. We suppress them for cleaner console output. |

### Why Not Seaborn, Plotly, or Other Libraries?

- **Seaborn** would simplify box plots and scatter plots, but it abstracts away the fine-grained control we need for custom annotations, dual-panel layouts, star markers, and badge overlays.
- **Plotly** produces interactive HTML charts, but our goal is static, print-ready PNG images at 180 DPI.
- **Altair** is great for declarative visualization but lacks the imperative control needed for custom text annotations and multi-figure compositions.

## 4. Analysis Pipeline Overview

The analysis follows a structured pipeline with 5 stages:

```
┌─────────────────────────────────────────────────────────────────┐
│  STAGE 1: DATA INGESTION & CLEANING                           │
│  • Read WHR_2025.xls via xlrd engine                          │
│  • Strip whitespace from column headers                       │
│  • Rename columns to short, Pythonic names                    │
│  • Assign 10 custom regions via REGION_MAP dictionary         │
│  • Generate rank column from pre-sorted order                 │
├─────────────────────────────────────────────────────────────────┤
│  STAGE 2: DESCRIPTIVE ANALYSIS                                │
│  • Identify top 20, bottom 20, and top 30 countries           │
│  • Compute regional medians and distributions                 │
│  • Extract Türkiye's rank and score as a reference point      │
├─────────────────────────────────────────────────────────────────┤
│  STAGE 3: CORRELATION ANALYSIS                                │
│  • Pearson correlation of each factor vs happiness            │
│  • p-value significance testing                               │
│  • Linear regression trend fitting (GDP vs happiness)         │
├─────────────────────────────────────────────────────────────────┤
│  STAGE 4: PARADOX DETECTION                                   │
│  • Z-score standardization of GDP and happiness               │
│  • Residual analysis: paradox = z(happiness) − z(GDP)         │
│  • Identify top/bottom 10 outliers                            │
├─────────────────────────────────────────────────────────────────┤
│  STAGE 5: VISUALIZATION (6 publication-quality charts)        │
│  • Consistent theme: off-white palette, no top/right spines   │
│  • Region-aware color coding (10-color palette)               │
│  • Annotated labels, confidence intervals, significance stars │
│  • 180 DPI export for print-quality resolution                │
└─────────────────────────────────────────────────────────────────┘
```

### Why This Order?

We deliberately clean the data first, then add derived columns (region, rank), then compute statistics, and finally visualize. This ensures that every chart operates on the same cleaned, enriched DataFrame — no duplicated logic, no inconsistencies.

## 5. Design Decisions & Rationale

### 5.1 Why 10 Regions?

The original WHR dataset does **not** include a region column. We created a custom 10-region classification loosely based on the UN geoscheme:

| Region | # Countries | Notes |
|--------|:-----------:|-------|
| Western Europe | 21 | Includes Switzerland, Cyprus, Malta |
| Eastern Europe | 20 | Includes Russia, Ukraine, Greece (cultural proximity) |
| Latin America | 22 | Central America, South America, Caribbean |
| North America & ANZ | 4 | USA, Canada, Australia, New Zealand |
| Middle East & North Africa | 18 | Includes Türkiye, Israel |
| Sub-Saharan Africa | 30 | Largest region by country count |
| East Asia | 6 | China, Japan, Korea, Taiwan, HK, Mongolia |
| Southeast Asia | 9 | ASEAN nations |
| South Asia | 6 | India, Pakistan, Nepal, etc. |
| Central Asia | 7 | Former Soviet + Caucasus states |

**Why not use the WHR's own regional groupings?** Because the WHR doesn't provide one in this dataset. Our groupings allow for more granular regional analysis than a simple continent split.

### 5.2 Why These Specific Colors?

The 10-color palette uses **Material Design** colors:
- Each color is perceptually distinct in both hue and lightness
- They work reasonably well for the most common forms of color blindness
- They look professional on both screen and print

### 5.3 Why `alpha=0.88` for Bars?

Setting opacity to 88% (instead of 100%) creates a subtle translucency that:
- Prevents the bars from looking harsh or flat
- Allows gridlines to remain faintly visible through the bars
- Creates a modern, editorial feel

### 5.4 Why 180 DPI?

- **72 DPI** = screen resolution (too blurry for presentations)
- **150 DPI** = decent quality but text can look fuzzy at large sizes
- **180 DPI** = excellent balance of quality and file size (~150–260 KB per chart)
- **300 DPI** = print-quality but files become unnecessarily large for web/GitHub

### 5.5 Why `#FAFAF8` Background?

Pure white (`#FFFFFF`) is harsh on the eyes and looks clinical. The warm off-white `#FAFAF8` gives charts an elegant, editorial appearance — similar to what you'd see in The Economist or Financial Times infographics.

### 5.6 Why Remove Top and Right Spines?

Removing the top and right axis borders (`axes.spines.top = False`, `axes.spines.right = False`) is a widely-adopted practice in modern data visualization (Edward Tufte's "data-ink ratio" principle). It reduces visual clutter without losing any information.

### 5.7 Why Highlight Türkiye?

Türkiye is consistently annotated across all 6 charts as a **personal reference point**. This is a common technique in data journalism — anchoring an unfamiliar dataset to something personally relevant makes the data more relatable and engaging.

## 6. Chart-by-Chart Methodology

### Chart 1: Top 20 & Bottom 20 Rankings
- **Chart type:** Horizontal bar chart (dual-panel)
- **Why horizontal bars?** Country names are long strings. Horizontal bars allow full names to be displayed on the y-axis without rotation or truncation.
- **Why error bars?** The WHR provides 95% confidence intervals (upper/lower whiskers). Including them shows the uncertainty in each country's ranking — e.g., two countries with overlapping intervals may not be meaningfully different.
- **Why top 20 / bottom 20?** It's a classic "extremes" view. 20 is enough to see patterns (regional clusters, score distributions) without overwhelming the viewer.

### Chart 2: Factor Breakdown (Top 30)
- **Chart type:** Stacked horizontal bar chart
- **Why stacked bars?** They show both the total (bar length) and the composition (segment widths) simultaneously. This is the only chart type that answers "*why* is this country happy?" rather than just "*how* happy is it?"
- **Why top 30?** 20 feels too few to see variation in factor composition, 50 makes the chart unreadable. 30 is the sweet spot.
- **Why 7 segments?** These are the exact factors provided by the WHR methodology. The 7th (Dystopia + residual) represents the baseline.

### Chart 3: Regional Distribution
- **Chart type:** Box plot + jittered scatter ("box-swarm" hybrid)
- **Why box plots?** They efficiently summarize five statistics: median, Q1, Q3, whiskers, and outliers. Perfect for comparing distributions across groups.
- **Why overlay scatter dots?** Box plots hide the individual data points and sample size. Overlaying jittered dots reveals the actual distribution shape and any clusters or gaps.
- **Why `np.random.normal(i, 0.06)` for jitter?** A standard deviation of 0.06 creates enough horizontal spread to prevent dot overlap while keeping dots visually associated with their box. Too much jitter (>0.15) would make dots bleed into adjacent boxes.
- **Why order by median?** Sorting regions by descending median happiness creates a natural left-to-right narrative: "these regions are happiest, these are least happy."

### Chart 4: GDP vs Happiness Scatter
- **Chart type:** Scatter plot with linear trend line
- **Why Pearson's r?** The Pearson correlation coefficient measures **linear** association, which is appropriate here because the GDP column is already log-transformed by the WHR. A log-transformed GDP vs happiness relationship is approximately linear.
- **Why `np.polyfit(degree=1)`?** This fits a simple linear regression (y = mx + b) to overlay a trend line. We chose degree 1 (linear) rather than polynomial because the theoretical relationship (after log-transform) is expected to be linear.
- **Why label 12 specific countries?** We hand-picked countries that represent: the top (Finland), the bottom (Afghanistan), a major economy (USA, China), a paradox case (Costa Rica — high happiness despite moderate GDP), and the personal reference point (Türkiye).

### Chart 5: Factor Correlations
- **Chart type:** Horizontal bar chart of Pearson r-values
- **Why Pearson correlation?** It's the standard measure for linear relationships between two continuous variables. All our variables are continuous and approximately normally distributed after the WHR's preprocessing.
- **Why significance stars?** Following academic convention: `*** p<0.001`, `** p<0.01`, `* p<0.05`. This tells the reader whether the correlation is statistically meaningful or could be due to chance.
- **Why `.fillna(0)` for missing values?** A few countries have missing data for certain factors (e.g., Generosity). Filling with 0 is conservative — it treats missing data as "no contribution," which is the safest assumption without imputation.
- **Why green/red coloring?** Positive correlations are green ("good"), negative correlations are red ("bad"). This is a universally understood color convention.

### Chart 6: Happiness Paradox
- **Chart type:** Dual-panel paired horizontal bar chart
- **Why z-scores?** GDP contribution and happiness scores are on different scales. Z-score standardization (mean=0, std=1) makes them directly comparable: a country with `happy_z > gdp_z` is "happier than its wealth predicts."
- **Why `paradox = happy_z − gdp_z`?** This is a simple residual analysis. A large positive paradox means happiness exceeds what GDP alone would predict (non-economic factors compensate). A large negative paradox means wealth isn't translating to well-being.
- **Why top/bottom 10?** 10 outliers on each side is enough to identify clear patterns (Latin American overachievers, Gulf state underachievers) without noise.

---

# PART II — STEP-BY-STEP CODE WALKTHROUGH

Below, we walk through every line of `analysis.py` in executable cells, explaining what each block does, why it's written that way, and what output to expect.

---

## Step 1: Imports & Environment Setup

We begin by importing every library we'll need and configuring the environment.

**Key decisions:**
- `sys.stdout.reconfigure(encoding='utf-8')` — Windows consoles (especially with Turkish locale cp1254) can't print the 🇹🇷 flag emoji, causing a `UnicodeEncodeError`. This line forces UTF-8 encoding on stdout.
- `warnings.filterwarnings('ignore')` — Matplotlib emits warnings about font substitutions and deprecated APIs. These are noise for our analysis.
- `os.makedirs('results', exist_ok=True)` — Creates the output directory idempotently. `exist_ok=True` prevents an error if the directory already exists.

In [ ]:
import os
import sys

# Force UTF-8 encoding on stdout to avoid UnicodeEncodeError on Windows
# consoles that default to a legacy code page (e.g., cp1254 for Turkish locale).
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import zscore
import warnings

# Suppress non-critical warnings to keep output clean.
warnings.filterwarnings('ignore')

# Ensure the output directory exists before we try to save any figures.
os.makedirs('results', exist_ok=True)

print('All libraries imported successfully.')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

## Step 2: Global Matplotlib Theme

Instead of styling each chart individually, we set a **global theme** using `plt.rcParams.update()`. This ensures visual consistency across all 6 charts.

**Line-by-line explanation:**

| Parameter | Value | Why |
|-----------|-------|-----|
| `font.family` | `'DejaVu Sans'` | A clean sans-serif font bundled with matplotlib. Available on all platforms without extra installation. |
| `axes.spines.top` | `False` | Removes the top border of charts — follows Edward Tufte's "minimize non-data ink" principle. |
| `axes.spines.right` | `False` | Removes the right border. Same rationale as above. |
| `figure.facecolor` | `'#FAFAF8'` | Warm off-white background. Pure white is harsh; this gives an editorial feel. |
| `axes.facecolor` | `'#FAFAF8'` | Matches the figure background so there's no visible "border" around the chart area. |
| `axes.grid` | `True` | Subtle gridlines help readers trace values to the axis. |
| `grid.color` | `'#E8E8E3'` | Very light gray — visible enough to be useful, subtle enough not to dominate. |
| `grid.linewidth` | `0.6` | Thinner than default (1.0). Keeps gridlines subordinate to the data. |
| `axes.labelcolor` | `'#333333'` | Dark gray instead of pure black. Softer on the eyes. |
| `xtick.color` / `ytick.color` | `'#555555'` | Even lighter gray for tick labels — they're reference information, not primary content. |
| `text.color` | `'#222222'` | Near-black for general text. Maintains readability without the harshness of `#000000`. |

In [ ]:
# Global Matplotlib Theme
# This dictionary overrides matplotlib's default style for ALL subsequent charts.
plt.rcParams.update({
    'font.family':        'DejaVu Sans',   # Clean sans-serif font, bundled with matplotlib
    'axes.spines.top':    False,            # Remove top border (Tufte principle)
    'axes.spines.right':  False,            # Remove right border
    'figure.facecolor':   '#FAFAF8',        # Warm off-white background
    'axes.facecolor':     '#FAFAF8',        # Match axes to figure
    'axes.grid':          True,             # Enable subtle gridlines
    'grid.color':         '#E8E8E3',        # Light gray grid
    'grid.linewidth':     0.6,              # Thin gridlines
    'axes.labelcolor':    '#333333',         # Dark gray axis labels
    'xtick.color':        '#555555',         # Muted tick labels
    'ytick.color':        '#555555',
    'text.color':         '#222222',         # Near-black text
})

print('Matplotlib theme configured.')

## Step 3: Region Color Palette

We define a dictionary mapping each of our 10 world regions to a specific hex color. These colors come from Google's **Material Design** color palette, which is designed for:

- **High perceptual distinctiveness** — each color is visually unique
- **Accessibility** — reasonable contrast for common forms of color blindness
- **Aesthetic harmony** — the colors look professional together

This dictionary is referenced in every chart that uses color-coding by region.

In [ ]:
# Region Color Palette — Material Design colors for 10 world regions
REGION_COLORS = {
    'Western Europe':              '#2196F3',   # Blue
    'North America & ANZ':         '#4CAF50',   # Green
    'Latin America':               '#FF9800',   # Orange
    'Eastern Europe':              '#9C27B0',   # Purple
    'East Asia':                   '#F44336',   # Red
    'South Asia':                  '#FF5722',   # Deep Orange
    'Middle East & North Africa':  '#795548',   # Brown
    'Sub-Saharan Africa':          '#607D8B',   # Blue Gray
    'Southeast Asia':              '#009688',   # Teal
    'Central Asia':                '#E91E63',   # Pink
}

# Display the palette
for region, color in REGION_COLORS.items():
    print(f'  {color}  →  {region}')

## Step 4: Country-to-Region Mapping

The WHR dataset does not include a region column, so we manually assign each of the 147 countries to one of our 10 regions. This mapping is stored as a Python dictionary for O(1) lookup performance.

**Notable classification decisions:**
- **Australia & New Zealand** → grouped with North America ("ANZ") rather than Asia-Pacific, because their happiness profiles and social structures more closely resemble Western/North American countries.
- **Türkiye** → Middle East & North Africa, reflecting its geopolitical and cultural positioning in the WHR context.
- **Greece** → Eastern Europe, based on its economic and happiness profile being closer to Eastern European neighbors.
- **Caucasus nations** (Georgia, Armenia, Azerbaijan) → Central Asia, as their profiles differ from both Europe and the Middle East.
- **Russia & Ukraine** → Eastern Europe, following standard geopolitical grouping.

In [ ]:
# Country-to-Region Mapping
# Each of the 147 countries is assigned to one of 10 macro-regions.
REGION_MAP = {
    # Western Europe (21 countries)
    'Finland':'Western Europe','Denmark':'Western Europe','Iceland':'Western Europe',
    'Sweden':'Western Europe','Netherlands':'Western Europe','Norway':'Western Europe',
    'Luxembourg':'Western Europe','Switzerland':'Western Europe',
    'Belgium':'Western Europe','Ireland':'Western Europe','Austria':'Western Europe',
    'Germany':'Western Europe','United Kingdom':'Western Europe',
    'France':'Western Europe','Spain':'Western Europe','Italy':'Western Europe',
    'Portugal':'Western Europe','Malta':'Western Europe','Cyprus':'Western Europe',
    
    # North America & ANZ (4 countries)
    'Australia':'North America & ANZ','New Zealand':'North America & ANZ',
    'Canada':'North America & ANZ','United States':'North America & ANZ',
    
    # Eastern Europe (20 countries)
    'Slovenia':'Eastern Europe','Czechia':'Eastern Europe','Lithuania':'Eastern Europe',
    'Poland':'Eastern Europe','Serbia':'Eastern Europe','Romania':'Eastern Europe',
    'Estonia':'Eastern Europe','Slovakia':'Eastern Europe','Latvia':'Eastern Europe',
    'Hungary':'Eastern Europe','Croatia':'Eastern Europe','Bulgaria':'Eastern Europe',
    'North Macedonia':'Eastern Europe','Bosnia and Herzegovina':'Eastern Europe',
    'Kosovo':'Eastern Europe','Albania':'Eastern Europe','Montenegro':'Eastern Europe',
    'Republic of Moldova':'Eastern Europe','Greece':'Eastern Europe',
    'Russian Federation':'Eastern Europe','Ukraine':'Eastern Europe',
    
    # Latin America (22 countries)
    'Costa Rica':'Latin America','Mexico':'Latin America','Uruguay':'Latin America',
    'Brazil':'Latin America','El Salvador':'Latin America','Argentina':'Latin America',
    'Guatemala':'Latin America','Chile':'Latin America','Nicaragua':'Latin America',
    'Paraguay':'Latin America','Panama':'Latin America','Colombia':'Latin America',
    'Ecuador':'Latin America','Honduras':'Latin America','Peru':'Latin America',
    'Bolivia':'Latin America','Dominican Republic':'Latin America',
    'Jamaica':'Latin America','Trinidad and Tobago':'Latin America',
    'Venezuela':'Latin America','Belize':'Latin America',
    
    # Middle East & North Africa (18 countries)
    'United Arab Emirates':'Middle East & North Africa',
    'Kuwait':'Middle East & North Africa','Saudi Arabia':'Middle East & North Africa',
    'Bahrain':'Middle East & North Africa','Oman':'Middle East & North Africa',
    'Israel':'Middle East & North Africa','State of Palestine':'Middle East & North Africa',
    'Libya':'Middle East & North Africa','Algeria':'Middle East & North Africa',
    'Iran':'Middle East & North Africa','Iraq':'Middle East & North Africa',
    'Morocco':'Middle East & North Africa','Tunisia':'Middle East & North Africa',
    'Egypt':'Middle East & North Africa','Jordan':'Middle East & North Africa',
    'Lebanon':'Middle East & North Africa','Yemen':'Middle East & North Africa',
    'Mauritania':'Middle East & North Africa',
    'Türkiye':'Middle East & North Africa',
    
    # East Asia (6 countries)
    'Taiwan Province of China':'East Asia','Japan':'East Asia',
    'Republic of Korea':'East Asia','China':'East Asia',
    'Hong Kong SAR of China':'East Asia','Mongolia':'East Asia',
    
    # Southeast Asia (9 countries)
    'Singapore':'Southeast Asia','Thailand':'Southeast Asia',
    'Malaysia':'Southeast Asia','Viet Nam':'Southeast Asia',
    'Philippines':'Southeast Asia','Indonesia':'Southeast Asia',
    'Lao PDR':'Southeast Asia','Cambodia':'Southeast Asia','Myanmar':'Southeast Asia',
    
    # Central Asia (7 countries)
    'Kazakhstan':'Central Asia','Uzbekistan':'Central Asia',
    'Kyrgyzstan':'Central Asia','Tajikistan':'Central Asia',
    'Azerbaijan':'Central Asia','Georgia':'Central Asia','Armenia':'Central Asia',
    
    # South Asia (6 countries)
    'India':'South Asia','Pakistan':'South Asia','Nepal':'South Asia',
    'Bangladesh':'South Asia','Sri Lanka':'South Asia','Afghanistan':'South Asia',
    
    # Sub-Saharan Africa (30 countries)
    'Nigeria':'Sub-Saharan Africa','Ghana':'Sub-Saharan Africa',
    'Kenya':'Sub-Saharan Africa','Ethiopia':'Sub-Saharan Africa',
    'Tanzania':'Sub-Saharan Africa','Uganda':'Sub-Saharan Africa',
    'Cameroon':'Sub-Saharan Africa','Senegal':'Sub-Saharan Africa',
    'Zambia':'Sub-Saharan Africa','Zimbabwe':'Sub-Saharan Africa',
    'Botswana':'Sub-Saharan Africa','South Africa':'Sub-Saharan Africa',
    'Mauritius':'Sub-Saharan Africa','Mozambique':'Sub-Saharan Africa',
    'Gabon':'Sub-Saharan Africa','Congo':'Sub-Saharan Africa',
    "Côte d'Ivoire":'Sub-Saharan Africa','Guinea':'Sub-Saharan Africa',
    'Namibia':'Sub-Saharan Africa','Niger':'Sub-Saharan Africa',
    'Chad':'Sub-Saharan Africa','Burkina Faso':'Sub-Saharan Africa',
    'Benin':'Sub-Saharan Africa','Somalia':'Sub-Saharan Africa',
    'Mali':'Sub-Saharan Africa','Togo':'Sub-Saharan Africa',
    'Liberia':'Sub-Saharan Africa','Madagascar':'Sub-Saharan Africa',
    'Eswatini':'Sub-Saharan Africa','Lesotho':'Sub-Saharan Africa',
    'Comoros':'Sub-Saharan Africa','DR Congo':'Sub-Saharan Africa',
    'Malawi':'Sub-Saharan Africa','Sierra Leone':'Sub-Saharan Africa',
}

print(f'Mapped {len(REGION_MAP)} countries to {len(set(REGION_MAP.values()))} regions.')

## Step 5: Data Loading & Preparation

This is the most critical data engineering step. We:

1. **Read the Excel file** using `pd.read_excel()` with `engine='xlrd'` because the file is in legacy `.xls` format.
2. **Strip whitespace** from column headers — a defensive measure, since Excel files sometimes have trailing spaces in headers that cause silent bugs.
3. **Rename columns** to short, Pythonic names. This makes all downstream code cleaner (e.g., `df['happiness']` instead of `df['Ladder score']`).
4. **Map countries to regions** using our `REGION_MAP` dictionary. `fillna('Other')` catches any countries we didn't explicitly map.
5. **Assign ranks** — the dataset arrives pre-sorted by Ladder score (highest first), so a simple `range(1, N+1)` gives us the correct global rank.

In [ ]:
# Load the dataset
# engine='xlrd' is required because the file is legacy .xls (not .xlsx)
df = pd.read_excel('WHR_2025.xls', engine='xlrd')

# Strip whitespace from column headers to prevent silent matching failures
df.columns = df.columns.str.strip()

# Rename columns to short, Pythonic labels for easier downstream use
df = df.rename(columns={
    'Country name':                               'country',
    'Ladder score':                               'happiness',
    'upperwhisker':                               'upper',
    'lowerwhisker':                               'lower',
    'Explained by: Log GDP per capita':           'gdp',
    'Explained by: Social support':               'social',
    'Explained by: Healthy life expectancy':      'life_exp',
    'Explained by: Freedom to make life choices': 'freedom',
    'Explained by: Generosity':                   'generosity',
    'Explained by: Perceptions of corruption':    'corruption',
    'Dystopia + residual':                        'dystopia',
})

# Assign region labels using our custom mapping
df['region'] = df['country'].map(REGION_MAP).fillna('Other')

# The dataset is pre-sorted by happiness (descending), so sequential indices = ranks
df['rank'] = range(1, len(df) + 1)

print(f'Loaded {len(df)} countries')
print(f'Columns: {list(df.columns)}')
print(f'\nScore range: {df["happiness"].min():.3f} — {df["happiness"].max():.3f}')
print(f'Regions: {df["region"].nunique()} unique')
print(f'\nTop 10 countries:')
df[['rank', 'country', 'happiness', 'region']].head(10)

### Quick Data Quality Check

Let's verify the data looks correct before proceeding:

In [ ]:
# Data quality checks
print('=== Data Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Descriptive Statistics ===')
df[['happiness', 'gdp', 'social', 'life_exp', 'freedom', 'generosity', 'corruption']].describe().round(3)

## Step 6: Türkiye Reference Point

We extract Türkiye's rank and score **once** at the top of the script and store them in global variables. This way, every chart can display a consistent callout badge without repeating the lookup logic.

**Why do this upfront?**
- Avoids code duplication (DRY principle)
- If the dataset changes, all charts automatically update
- The f-string label format (`🇹🇷 Türkiye  #94/147 — 5.262`) is reused identically across all 6 charts

In [ ]:
# Extract Türkiye's data as a reference point for all charts
tr = df[df['country'] == 'Türkiye'].iloc[0]
TR_RANK  = int(tr['rank'])
TR_SCORE = float(tr['happiness'])
TR_LABEL = f'🇹🇷 Türkiye  #{TR_RANK}/147 — {TR_SCORE:.3f}'

print(f'Türkiye ranks #{TR_RANK} out of 147 countries')
print(f'Happiness score: {TR_SCORE:.3f}')
print(f'Region: {tr["region"]}')
print(f'\nLabel used on all charts: {TR_LABEL}')

# Show Türkiye's full factor breakdown
print('\n=== Türkiye Factor Breakdown ===')
for factor in ['gdp', 'social', 'life_exp', 'freedom', 'generosity', 'corruption', 'dystopia']:
    print(f'  {factor:12s}: {tr[factor]:.3f}')

---

## Step 7: Chart 1 — Top 20 & Bottom 20 Rankings

### What This Chart Shows
Side-by-side horizontal bar charts of the 20 happiest and 20 unhappiest countries, with 95% confidence interval error bars.

### Code Walkthrough

**Data preparation:**
- `df.head(20)` / `df.tail(20)` — since the DataFrame is pre-sorted by happiness, `head` gives us the top and `tail` gives the bottom.
- `[::-1]` — reverses the order so that the highest-ranked country appears at the top of the horizontal bar chart (matplotlib draws from bottom to top by default).

**Figure setup:**
- `plt.subplots(1, 2, figsize=(16, 8))` — creates two side-by-side panels. 16×8 inches is wide enough for two bar charts with full country names.
- `fig.patch.set_facecolor('#FAFAF8')` — explicitly sets the figure background (the `rcParams` setting sometimes doesn't apply to saved files).

**Bar coloring:**
- Each bar's color is determined by looking up the country's region in `REGION_COLORS`. This adds a second layer of information without needing a separate legend.

**Error bars:**
- `ax.errorbar(xerr=[lower_error, upper_error])` — the asymmetric error bars use the WHR's confidence intervals. This is important because it shows that some country rankings are statistically interchangeable.

**Value labels:**
- We place the exact score (e.g., `7.736`) at the end of each bar using `ax.text()`. This lets readers compare precise values without needing to trace lines to the axis.

In [ ]:
# CHART 1 — Top 20 & Bottom 20 Happiness Rankings

top20 = df.head(20).copy()
bot20 = df.tail(20).copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor('#FAFAF8')
for ax in [ax1, ax2]:
    ax.set_facecolor('#FAFAF8')

# --- Left panel: Top 20 Happiest ---
top_colors = [REGION_COLORS.get(r, '#999') for r in top20['region']]
bars1 = ax1.barh(top20['country'][::-1], top20['happiness'][::-1],
                 color=top_colors[::-1], alpha=0.88,
                 edgecolor='white', linewidth=0.8)

# 95% confidence interval error bars
ax1.errorbar(
    top20['happiness'][::-1], range(len(top20)),
    xerr=[top20['happiness'][::-1].values - top20['lower'][::-1].values,
          top20['upper'][::-1].values - top20['happiness'][::-1].values],
    fmt='none', color='#555', capsize=3, linewidth=1, zorder=4
)

# Value labels at bar ends
for bar, val in zip(bars1, top20['happiness'][::-1]):
    ax1.text(bar.get_width() + 0.04, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=8.5, fontweight='bold', color='#333')
ax1.set_xlim(5.5, 8.6)
ax1.set_xlabel('Happiness Score (Ladder Score)', fontsize=10)
ax1.set_title('🌟 Top 20 Happiest Countries\n2025 World Happiness Report',
              fontsize=12, fontweight='bold', pad=12)

# --- Right panel: Bottom 20 Unhappiest ---
bot_colors = [REGION_COLORS.get(r, '#999') for r in bot20['region']]
bars2 = ax2.barh(bot20['country'][::-1], bot20['happiness'][::-1],
                 color=bot_colors[::-1], alpha=0.88,
                 edgecolor='white', linewidth=0.8)
ax2.errorbar(
    bot20['happiness'][::-1], range(len(bot20)),
    xerr=[bot20['happiness'][::-1].values - bot20['lower'][::-1].values,
          bot20['upper'][::-1].values - bot20['happiness'][::-1].values],
    fmt='none', color='#555', capsize=3, linewidth=1, zorder=4
)
for bar, val in zip(bars2, bot20['happiness'][::-1]):
    ax2.text(bar.get_width() + 0.04, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=8.5, fontweight='bold', color='#333')
ax2.set_xlim(1.2, 4.8)
ax2.set_xlabel('Happiness Score (Ladder Score)', fontsize=10)
ax2.set_title('💔 Bottom 20 Unhappiest Countries\n2025 World Happiness Report',
              fontsize=12, fontweight='bold', pad=12)

# Shared region legend
legend_patches = [mpatches.Patch(color=c, label=r)
                  for r, c in REGION_COLORS.items()
                  if r in df['region'].unique()]
fig.legend(handles=legend_patches, loc='lower center', ncol=5,
           fontsize=8, title='Region', title_fontsize=9,
           bbox_to_anchor=(0.5, -0.04), framealpha=0.9)

# Source citation
ax1.text(0.02, -0.07,
         'Source: World Happiness Report 2025 | Error bars show 95% confidence interval',
         transform=ax1.transAxes, fontsize=7.5, color='#888', style='italic')

# Türkiye callout badge
fig.text(0.5, -0.06, TR_LABEL,
         ha='center', fontsize=10, fontweight='bold',
         color='#C62828',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E1',
                   edgecolor='#E53935', linewidth=1.2))

plt.tight_layout()
plt.savefig('results/happiness_rankings_top_bottom_20.png', dpi=180,
            bbox_inches='tight', facecolor='#FAFAF8')
plt.show()
print('Chart 1 saved → results/happiness_rankings_top_bottom_20.png')

---

## Step 8: Chart 2 — Factor Breakdown (Top 30)

### What This Chart Shows
A stacked horizontal bar chart decomposing each of the top 30 countries' happiness scores into 7 factors.

### Code Walkthrough

**Why top 30 instead of top 20?**
- With 30 countries, we see more variation in factor composition. For example, countries ranked 20–30 may have very different factor profiles than the top 10.

**Stacking mechanism:**
- We use a `lefts` accumulator initialized to `np.zeros(30)`. For each factor, we draw bars starting at `left=lefts`, then add the current factor's values to `lefts`. This creates the stacking effect.
- `fillna(0)` ensures that missing values don't break the stacking.

**The 7 factors and their colors:**
- GDP → Blue (`#1976D2`)
- Social Support → Green (`#388E3C`)
- Life Expectancy → Orange (`#F57C00`)
- Freedom → Purple (`#7B1FA2`)
- Generosity → Deep Orange (`#E64A19`)
- Low Corruption → Teal (`#00796B`)
- Baseline → Light Gray (`#CFD8DC`) — intentionally muted because it's a residual, not a measured factor.

**Why `edgecolor='white', linewidth=0.4`?**
- Thin white borders between segments make the stacking visually clear. Without them, adjacent segments of similar colors would blend together.

In [ ]:
# CHART 2 — Factor Breakdown for the Top 30 Countries

# Reverse order so #1 appears at the top of the chart
top30 = df.head(30).copy().iloc[::-1].reset_index(drop=True)

# Factor columns, labels, and colors
factors     = ['gdp','social','life_exp','freedom','generosity','corruption','dystopia']
factor_lbls = ['GDP','Social Support','Life Expectancy','Freedom','Generosity','Low Corruption','Baseline']
factor_cols = ['#1976D2','#388E3C','#F57C00','#7B1FA2','#E64A19','#00796B','#CFD8DC']

fig, ax = plt.subplots(figsize=(13, 11))
fig.patch.set_facecolor('#FAFAF8')
ax.set_facecolor('#FAFAF8')

# Build stacked bars with running left-offset
lefts = np.zeros(len(top30))
for fac, lbl, col in zip(factors, factor_lbls, factor_cols):
    vals = top30[fac].fillna(0).values
    ax.barh(top30['country'], vals, left=lefts, color=col,
            label=lbl, alpha=0.88, edgecolor='white', linewidth=0.4)
    lefts += vals

# Total score labels at bar ends
for i, (_, row) in enumerate(top30.iterrows()):
    ax.text(row['happiness'] + 0.05, i, f"{row['happiness']:.3f}",
            va='center', fontsize=8, fontweight='bold', color='#333')

ax.set_xlabel('Happiness Score Contribution', fontsize=11)
ax.set_title("What Drives Happiness in the World's Top 30 Countries?\nFactor Breakdown — WHR 2025",
             fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9, edgecolor='#ddd')
ax.set_xlim(0, 10.5)
ax.text(0.01, -0.04,
        'Source: World Happiness Report 2025 | Each segment shows factor contribution.',
        transform=ax.transAxes, fontsize=7.5, color='#888', style='italic')

# Türkiye badge below the title
fig.text(0.5, 0.925, TR_LABEL,
         ha='center', fontsize=9, fontweight='bold',
         color='#C62828',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E1',
                   edgecolor='#E53935', linewidth=1.2))

plt.tight_layout()
plt.savefig('results/happiness_factor_breakdown_top30.png', dpi=180,
            bbox_inches='tight', facecolor='#FAFAF8')
plt.show()
print('Chart 2 saved → results/happiness_factor_breakdown_top30.png')

---

## Step 9: Chart 3 — Regional Distribution

### What This Chart Shows
Box plots for each of the 10 world regions, overlaid with individual country data points, showing the spread and central tendency of happiness scores within each region.

### Code Walkthrough

**Region ordering:**
- `df.groupby('region')['happiness'].median().sort_values(ascending=False)` — we sort regions by their median happiness so the viewer immediately sees the hierarchy.

**Box plot configuration:**
- `patch_artist=True` — fills the boxes with color (default is hollow).
- `medianprops=dict(color='white', linewidth=2)` — the median line is white and thick, creating high contrast against the colored box.

**Jittered scatter overlay:**
- `np.random.normal(i, 0.06, size=n)` — adds small random horizontal offsets. The standard deviation of 0.06 creates visible spread without dots crossing into adjacent boxes.
- `alpha=0.45` — semi-transparent dots prevent occlusion when countries overlap.
- `s=22` — small dot size, since dots are supplementary to the box plot.

**Türkiye highlight:**
- We detect Türkiye's region automatically and place a red star marker (`marker='*'`, `s=90`) at its position, with an arrow annotation.

**World average line:**
- `ax.axhline(world_avg)` — a dashed red line at the global mean provides an immediate reference: "is this region above or below the world average?"

**Sample sizes:**
- `n=X` labels below each box tell the reader how many countries are in each region. This is important because a box plot for 4 countries (North America & ANZ) is less reliable than one for 30 countries (Sub-Saharan Africa).

In [ ]:
# CHART 3 — Regional Distribution (Box + Swarm Plot)

# Sort regions by median happiness (descending)
region_order = (df.groupby('region')['happiness']
                  .median()
                  .sort_values(ascending=False)
                  .index.tolist())
region_order = [r for r in region_order if r != 'Other']

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#FAFAF8')
ax.set_facecolor('#FAFAF8')

# Prepare data in region order
data_by_region = [df[df['region'] == r]['happiness'].values for r in region_order]
bp = ax.boxplot(data_by_region, vert=True, patch_artist=True,
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                capprops=dict(linewidth=1.2),
                flierprops=dict(marker='o', markersize=4, alpha=0.5))

# Color boxes by region
for patch, region in zip(bp['boxes'], region_order):
    patch.set_facecolor(REGION_COLORS.get(region, '#999'))
    patch.set_alpha(0.82)

# Jittered scatter overlay
for i, (region, box_data) in enumerate(zip(region_order, data_by_region), 1):
    x = np.random.normal(i, 0.06, size=len(box_data))
    ax.scatter(x, box_data, alpha=0.45,
               color=REGION_COLORS.get(region, '#999'), s=22, zorder=3)
    
    # Highlight Türkiye in its region
    if region == df[df['country'] == 'Türkiye']['region'].values[0]:
        tr_idx = i
        ax.scatter([i], [TR_SCORE], color='#E53935', s=90, zorder=6,
                   marker='*', edgecolors='white', linewidths=0.5)
        ax.annotate(f'#{TR_RANK} Türkiye\n{TR_SCORE:.3f}',
                    xy=(i, TR_SCORE),
                    xytext=(i + 0.35, TR_SCORE + 0.12),
                    fontsize=8, fontweight='bold', color='#C62828',
                    arrowprops=dict(arrowstyle='->', color='#E53935', lw=1))

# Sample sizes
for i, (region, d) in enumerate(zip(region_order, data_by_region), 1):
    ax.text(i, df['happiness'].min() - 0.15, f'n={len(d)}',
            ha='center', fontsize=8, color='#888')

ax.set_xticks(range(1, len(region_order)+1))
ax.set_xticklabels(region_order, rotation=22, ha='right', fontsize=9)
ax.set_ylabel('Happiness Score (Ladder Score)', fontsize=11)
ax.set_title('How Happy Is Each Region of the World?\nDistribution by Region — WHR 2025',
             fontsize=14, fontweight='bold', pad=15)

# World average reference line
world_avg = df['happiness'].mean()
ax.axhline(world_avg, color='#E53935', linestyle='--', linewidth=1.2, alpha=0.7)
ax.text(len(region_order) + 0.3, world_avg,
        f'World avg\n{world_avg:.2f}', fontsize=8, color='#E53935', va='center')

ax.text(0.01, -0.12,
        'Source: World Happiness Report 2025 | Each dot is a country. Box shows median and IQR.',
        transform=ax.transAxes, fontsize=7.5, color='#888', style='italic')

plt.tight_layout()
plt.savefig('results/happiness_regional_distribution.png', dpi=180,
            bbox_inches='tight', facecolor='#FAFAF8')
plt.show()
print('Chart 3 saved → results/happiness_regional_distribution.png')

---

## Step 10: Chart 4 — GDP vs Happiness Scatter

### What This Chart Shows
A scatter plot of GDP contribution vs Happiness Score for all 147 countries, with a linear trend line and Pearson's *r*.

### Code Walkthrough

**Why a scatter plot?**
- It's the standard visualization for exploring the relationship between two continuous variables. Each dot is a country, positioned by its GDP (x-axis) and happiness (y-axis).

**Region-by-region plotting:**
- We loop through each region and create a separate scatter series. This gives us an automatic legend with one entry per region.

**Pearson correlation:**
- `stats.pearsonr(df['gdp'], df['happiness'])` returns both the correlation coefficient (*r*) and the p-value.
- We display *r* in the chart title so readers immediately see the strength of the linear relationship.

**Trend line:**
- `np.polyfit(df['gdp'], df['happiness'], 1)` fits a degree-1 (linear) polynomial: y = mx + b.
- `np.linspace(min, max, 200)` creates 200 evenly-spaced x-values for a smooth line.
- The trend line is dashed (`'--'`), semi-transparent (`alpha=0.45`), and behind the scatter points (`zorder=2`) to avoid visual interference.

**Hand-picked labels:**
- We annotate 12 countries that represent interesting reference points. This transforms the chart from an anonymous cloud of dots into a meaningful narrative.

**Türkiye star marker:**
- A larger star marker (`marker='*'`, `s=100`) with white edge makes Türkiye instantly findable.

In [ ]:
# CHART 4 — GDP vs Happiness Scatter Plot

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('#FAFAF8')
ax.set_facecolor('#FAFAF8')

# Plot each region as a separate scatter series
for region in df['region'].unique():
    if region == 'Other':
        continue
    sub = df[df['region'] == region]
    ax.scatter(sub['gdp'], sub['happiness'],
               c=REGION_COLORS.get(region, '#999'),
               s=65, alpha=0.82, edgecolors='white',
               linewidths=0.6, label=region, zorder=3)

# Pearson correlation and linear trend line
r_val, p_val = stats.pearsonr(df['gdp'], df['happiness'])
m, b = np.polyfit(df['gdp'], df['happiness'], 1)
xline = np.linspace(df['gdp'].min(), df['gdp'].max(), 200)
ax.plot(xline, m*xline + b, color='#333', linewidth=1.8,
        linestyle='--', alpha=0.45, zorder=2)

# Annotate notable countries
highlights = ['Finland','United States','Afghanistan','Lebanon','India',
              'China','Japan','Costa Rica','Zimbabwe','Singapore','Brazil','Türkiye']
for _, row in df[df['country'].isin(highlights)].iterrows():
    label = f"#{int(row['rank'])} {row['country']}" if row['country'] == 'Türkiye' else row['country']
    color = '#C62828' if row['country'] == 'Türkiye' else '#333'
    weight = 'bold' if row['country'] == 'Türkiye' else 'normal'
    ax.annotate(label, xy=(row['gdp'], row['happiness']),
                xytext=(4, 3), textcoords='offset points',
                fontsize=8, color=color, alpha=0.9, fontweight=weight)

# Türkiye star marker
tr_row = df[df['country'] == 'Türkiye'].iloc[0]
ax.scatter([tr_row['gdp']], [tr_row['happiness']], color='#E53935',
           s=100, zorder=6, marker='*', edgecolors='white', linewidths=0.5)

ax.set_xlabel('GDP per Capita Contribution (Log scale)', fontsize=11)
ax.set_ylabel('Happiness Score (Ladder Score)', fontsize=11)
ax.set_title(f'Wealth vs Happiness: How Strong Is the Link?\n147 Countries — WHR 2025  |  r = {r_val:.2f}',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(title='Region', bbox_to_anchor=(1.02, 1), loc='upper left',
          fontsize=8.5, title_fontsize=9, framealpha=0.9, edgecolor='#ddd')
ax.text(0.02, 0.04,
        'Source: World Happiness Report 2025 | Dashed line = linear trend',
        transform=ax.transAxes, fontsize=7.5, color='#888', style='italic')

plt.tight_layout()
plt.savefig('results/happiness_vs_gdp_scatter.png', dpi=180,
            bbox_inches='tight', facecolor='#FAFAF8')
plt.show()
print(f'Chart 4 saved → results/happiness_vs_gdp_scatter.png')
print(f'Pearson r = {r_val:.4f}, p-value = {p_val:.2e}')

---

## Step 11: Chart 5 — Factor Correlations

### What This Chart Shows
A horizontal bar chart of Pearson correlation coefficients (*r*) for each of the 6 happiness factors vs the overall score, with statistical significance stars.

### Code Walkthrough

**Correlation computation:**
- We loop through each factor, compute `stats.pearsonr(factor, happiness)`, and collect the results in a list of dictionaries.
- `fillna(0)` handles missing values conservatively (treats as "no contribution").

**Sorting:**
- `sort_values('r', ascending=True)` — ascending sort puts the weakest correlation at the top and the strongest at the bottom. Since matplotlib draws bars from bottom to top, the strongest correlator ends up at the visual top. This creates a natural "most important → least important" reading order.

**Significance stars:**
- `*** p<0.001` — very strong evidence the correlation isn't due to chance (>99.9% confidence)
- `** p<0.01` — strong evidence (>99% confidence)
- `* p<0.05` — moderate evidence (>95% confidence)

**Color convention:**
- Green (`#4CAF50`) for positive correlations, red (`#F44336`) for negative. All factors in this dataset happen to be positively correlated, but the code handles the negative case for robustness.

**`ax.axvline(0)` — the zero line:**
- A vertical line at r=0 provides a visual anchor. Bars extending further right indicate stronger positive correlations.

In [ ]:
# CHART 5 — Factor-Happiness Correlation Coefficients

factor_names = {
    'gdp':        'GDP per Capita',
    'social':     'Social Support',
    'life_exp':   'Healthy Life Expectancy',
    'freedom':    'Freedom to Make Choices',
    'generosity': 'Generosity',
    'corruption': 'Low Corruption',
}

# Compute Pearson correlations
corr_rows = []
for col, name in factor_names.items():
    r, p = stats.pearsonr(df[col].fillna(0), df['happiness'])
    corr_rows.append({'Factor': name, 'r': r, 'p': p})

corr_df = pd.DataFrame(corr_rows).sort_values('r', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#FAFAF8')
ax.set_facecolor('#FAFAF8')

# Color: green for positive, red for negative
bar_colors = ['#4CAF50' if r > 0 else '#F44336' for r in corr_df['r']]
bars = ax.barh(corr_df['Factor'], corr_df['r'],
               color=bar_colors, alpha=0.88,
               edgecolor='white', linewidth=0.8)

# Annotate with r-value and significance stars
for bar, (_, row) in zip(bars, corr_df.iterrows()):
    sig = '***' if row['p'] < 0.001 else ('**' if row['p'] < 0.01 else '*')
    offset = 0.01 if row['r'] > 0 else -0.01
    ha = 'left' if row['r'] > 0 else 'right'
    ax.text(row['r'] + offset, bar.get_y() + bar.get_height()/2,
            f"r = {row['r']:.2f} {sig}",
            va='center', fontsize=10, fontweight='bold', color='#333', ha=ha)

ax.axvline(0, color='#333', linewidth=1)
ax.set_xlabel('Pearson Correlation Coefficient (r)', fontsize=11)
ax.set_title("What Drives Happiness the Most?\nCorrelation of Each Factor with Happiness Score — WHR 2025",
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(-0.2, 1.0)

ax.text(0.98, 0.04, '*** p<0.001   ** p<0.01   * p<0.05',
        transform=ax.transAxes, fontsize=8, color='#888', style='italic', ha='right')
ax.text(0.01, -0.08,
        'Source: World Happiness Report 2025',
        transform=ax.transAxes, fontsize=7.5, color='#888', style='italic')

# Türkiye badge next to x-axis label
ax.text(0.99, -0.08, TR_LABEL,
        transform=ax.transAxes, ha='right', va='top', fontsize=9, fontweight='bold',
        color='#C62828',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E1',
                  edgecolor='#E53935', linewidth=1.2))

plt.tight_layout()
plt.savefig('results/happiness_factor_correlations.png', dpi=180,
            bbox_inches='tight', facecolor='#FAFAF8')
plt.show()
print('Chart 5 saved → results/happiness_factor_correlations.png')

# Print the correlation table
print('\n=== Factor Correlation Summary ===')
for _, row in corr_df.sort_values('r', ascending=False).iterrows():
    sig = '***' if row['p'] < 0.001 else ('**' if row['p'] < 0.01 else '*')
    print(f"  {row['Factor']:30s}  r = {row['r']:.4f}  {sig}  (p = {row['p']:.2e})")

---

## Step 12: Chart 6 — Happiness Paradox

### What This Chart Shows
A dual-panel chart identifying countries that are significantly happier (or unhappier) than their GDP alone would predict.

### The Paradox Algorithm — Step by Step

1. **Z-score standardize GDP:** `z(GDP) = (GDP - mean) / std`
   - This transforms GDP to a standard scale where mean=0 and std=1.
   - A country with `z(GDP) = +1.5` has GDP 1.5 standard deviations above the global average.

2. **Z-score standardize happiness:** `z(happiness) = (happiness - mean) / std`
   - Same transformation for happiness scores.

3. **Compute the paradox score:** `paradox = z(happiness) − z(GDP)`
   - If `paradox > 0`: the country is happier than its GDP would predict.
   - If `paradox < 0`: the country is less happy than its GDP would predict.
   - The magnitude indicates how extreme the discrepancy is.

4. **Select extremes:** `nlargest(10)` and `nsmallest(10)` identify the 10 most extreme outliers on each side.

### Why Z-Scores?

GDP contribution (~0.0 to ~2.0) and happiness score (~1.7 to ~7.7) are on completely different scales. We can't directly compare them. Z-score standardization puts both on the same unitless scale, making subtraction meaningful.

### The `dual_bar()` Helper Function

This helper draws paired horizontal bars (GDP in blue, happiness in green/red) side by side. The visual gap between the two bars for each country directly shows the paradox — a big gap means a big discrepancy.

- `h = 0.35` — each bar is 0.35 units tall. Two bars at `y + h/2` and `y - h/2` create a clean pair without overlap.
- `alpha=0.65` — slightly more transparent than other charts, since we're overlaying two bars and need to see both clearly.

In [ ]:
# CHART 6 — The Happiness Paradox

# Standardize both variables to z-scores for direct comparison
df['gdp_z']   = zscore(df['gdp'])
df['happy_z'] = zscore(df['happiness'])
df['paradox'] = df['happy_z'] - df['gdp_z']

# Top 10 overachievers and underachievers
over  = df.nlargest(10, 'paradox')[['country','happiness','gdp','region','paradox']]
under = df.nsmallest(10, 'paradox')[['country','happiness','gdp','region','paradox']]

print('=== Top 10 Overachievers (Happier Than GDP Predicts) ===')
for _, r in over.iterrows():
    print(f"  {r['country']:25s}  happiness={r['happiness']:.3f}  gdp={r['gdp']:.3f}  paradox={r['paradox']:+.3f}")

print('\n=== Top 10 Underachievers (Less Happy Than GDP Predicts) ===')
for _, r in under.iterrows():
    print(f"  {r['country']:25s}  happiness={r['happiness']:.3f}  gdp={r['gdp']:.3f}  paradox={r['paradox']:+.3f}")

In [ ]:
# Visualize the paradox

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#FAFAF8')
for ax in [ax1, ax2]:
    ax.set_facecolor('#FAFAF8')

def dual_bar(ax, data, gdp_color, hap_color, title):
    """
    Draw paired horizontal bars for GDP and happiness side by side.
    The visual gap between bars shows the paradox magnitude.
    """
    countries = data['country'].values[::-1]
    gdp_vals  = data['gdp'].values[::-1]
    hap_vals  = data['happiness'].values[::-1]
    y = np.arange(len(countries))
    h = 0.35  # Bar height — two bars of 0.35 fit cleanly in a 1-unit slot
    ax.barh(y + h/2, gdp_vals, h, color=gdp_color,
            alpha=0.65, label='GDP Contribution', edgecolor='white')
    ax.barh(y - h/2, hap_vals, h, color=hap_color,
            alpha=0.65, label='Happiness Score', edgecolor='white')
    ax.set_yticks(y)
    ax.set_yticklabels(countries, fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.legend(fontsize=9, framealpha=0.9)

# Left panel: overachievers
dual_bar(ax1, over,  '#90CAF9', '#A5D6A7',
         '🌿 Happier Than Their Wealth Predicts\n(Punching above their GDP weight)')

# Right panel: underachievers
dual_bar(ax2, under, '#90CAF9', '#EF9A9A',
         '💰 Less Happy Than Their Wealth Predicts\n(Punching below their GDP weight)')

# Source citations
for ax in [ax1, ax2]:
    ax.text(0.01, -0.06,
            'Source: World Happiness Report 2025 | Paradox score = happiness z-score minus GDP z-score',
            transform=ax.transAxes, fontsize=7.5, color='#888', style='italic')

# Türkiye badge
fig.text(0.5, -0.04, TR_LABEL,
         ha='center', fontsize=10, fontweight='bold',
         color='#C62828',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E1',
                   edgecolor='#E53935', linewidth=1.2))

plt.suptitle('The Happiness Paradox: Countries That Defy the Wealth Rule — WHR 2025',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/happiness_paradox_wealth_outliers.png', dpi=180,
            bbox_inches='tight', facecolor='#FAFAF8')
plt.show()
print('Chart 6 saved → results/happiness_paradox_wealth_outliers.png')

---

## Summary

All 6 charts have been generated and saved to the `results/` directory:

| # | Chart | File | Key Insight |
|---|-------|------|-------------|
| 1 | Top 20 & Bottom 20 | `happiness_rankings_top_bottom_20.png` | Finland leads; Afghanistan trails |
| 2 | Factor Breakdown | `happiness_factor_breakdown_top30.png` | GDP and Social Support are largest segments |
| 3 | Regional Distribution | `happiness_regional_distribution.png` | Western Europe leads; high variation in Africa |
| 4 | GDP vs Happiness | `happiness_vs_gdp_scatter.png` | Strong positive correlation (r ≈ 0.76) |
| 5 | Factor Correlations | `happiness_factor_correlations.png` | Social Support > GDP > Life Exp > Freedom > Corruption > Generosity |
| 6 | Happiness Paradox | `happiness_paradox_wealth_outliers.png` | Latin America overachieves; Gulf states underachieve |

### Key Takeaways

1. **Money matters, but it's not everything.** GDP is one of the strongest predictors, but countries like Costa Rica and Mexico achieve high happiness with moderate GDP.
2. **Social bonds are paramount.** Social Support has the highest correlation with happiness across all 147 countries.
3. **Generosity barely correlates.** While generosity is morally valued, it shows the weakest statistical link to national happiness.
4. **Regional patterns are stark.** Western Europe and North America & ANZ consistently lead, while Sub-Saharan Africa and South Asia lag — but with significant within-region variation.
5. **The Happiness Paradox is real.** Culture, freedom, social cohesion, and governance can compensate for lower GDP — or fail to translate wealth into well-being.

---

*Notebook created for the World Happiness Report 2025 Analysis Project.*  
*Dataset: [Kaggle — WHR 2025](https://www.kaggle.com/datasets/rmarbun/world-happiness-report-2025/data)*

---

### Contact & Links

* **GitHub:** [AlicanKaya192/World-Happiness-Report-2025](https://github.com/AlicanKaya192/World-Happiness-Report-2025)
* **LinkedIn:** [Alican Kaya](https://www.linkedin.com/posts/alican-kaya-881650234_datascience-dataanalysis-datavisualization-share-7462927970479960065-tvB6?utm_source=share&utm_medium=member_desktop&rcm=ACoAADp44eEB1YdNxrN_TSOrs8Qc-E8c5nUsSu8)
* **Portfolio:** [alican-kaya.com](https://alican-kaya.com/)
